# 03 — LSTM (Recurrent) Network on MNIST

This notebook trains an **LSTM** (Long Short-Term Memory) recurrent network on MNIST.

LSTMs are typically used for **sequences** (text, time series). Here we treat each MNIST image as a **sequence of 28 rows**, each row being a vector of 28 pixel values.

**Architecture:**
```
Input (28×28) → treated as 28 time-steps of 28 features
  → LSTM(128) → LSTM(64) → Dense(10, softmax)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

## 1. Load & Preprocess Data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalise — keep shape (N, 28, 28): 28 time-steps, 28 features each
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f'Train: {x_train.shape}  → 28 time-steps, 28 features')
print(f'Test:  {x_test.shape}')

## 2. Build the LSTM Model

We stack two LSTM layers. The first returns sequences (`return_sequences=True`) so the second LSTM can process them.

In [ ]:
def build_lstm_model():
    model = keras.Sequential([
        # Input: (batch, 28 time-steps, 28 features)
        layers.LSTM(128, return_sequences=True, input_shape=(28, 28)),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ], name='LSTM_RNN')
    return model

model = build_lstm_model()
model.summary()

## 3. Compile & Train

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)
elapsed = time.time() - start
print(f'\nTotal training time: {elapsed:.1f}s')

## 4. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')
print(f'Total Params  : {model.count_params():,}')
print(f'Training Time : {elapsed:.1f}s')

## 5. Why does LSTM work here?

Even though images aren't sequences in the traditional sense, the LSTM can capture **vertical patterns** across rows.

For example, a '1' has a consistent vertical stroke that the LSTM can track row-by-row.

Let's visualise how we feed an image as a sequence:

In [ ]:
sample = x_train[0]  # Shape: (28, 28)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(sample, cmap='gray')
axes[0].set_title(f'Image (label: {y_train[0]})')
axes[0].axis('off')

axes[1].imshow(sample, cmap='gray', aspect='auto')
for row in range(28):
    axes[1].axhline(y=row - 0.5, color='cyan', linewidth=0.4, alpha=0.5)
axes[1].set_title('Each row = one time-step fed to LSTM')
axes[1].set_xlabel('28 pixel features')
axes[1].set_ylabel('Time-steps (rows)')

plt.tight_layout()
plt.show()

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('LSTM — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('LSTM — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
import json, os

os.makedirs('../results', exist_ok=True)
results = {
    'model': 'LSTM (RNN)',
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'total_params': model.count_params(),
    'training_time_s': round(elapsed, 1),
    'history': {
        'accuracy':     history.history['accuracy'],
        'val_accuracy': history.history['val_accuracy'],
        'loss':         history.history['loss'],
        'val_loss':     history.history['val_loss'],
    }
}

with open('../results/lstm_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to results/lstm_results.json')